# 01 Load Excel Files

* Author: Jeremiah Hansen
* Last Updated: 6/17/2026

This notebook will load data into the `LOCATION` and `ORDER_DETAIL` tables from Excel files.

This currently does not use Snowpark File Access as it doesn't yet work in Notebooks. So for now we copy the file locally first.

In [ ]:
# Import python packages
import sys
import logging

# Set up the logger
logger_name = 'demo_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

# Set default values for debugging
notebook_name = '01_load_excel_files.ipynb'
database_name = 'DEMO_DB'
schema_name = 'DEV_SCHEMA'
role_name = 'DEMO_ROLE'

# Override values with passed notebook arguments
if sys.argv[0].endswith('.ipynb'):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--database-name', type=str)
    parser.add_argument('--schema-name', type=str)
    parser.add_argument('--role-name', type=str)
    args, args_unknown = parser.parse_known_args()

    notebook_name = parser.prog  # same as argv[0]
    database_name = args.database_name or database_name
    schema_name = args.schema_name or schema_name
    role_name = args.role_name or role_name

# Get a Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set the default database and schema for the following cells
session.use_schema(f"{database_name}.{schema_name}")

# Set the role
# Needed when running EXECUTE NOTEBOOK PROJECT directly (since it ignores session context and uses the user's default role)
session.use_role(role_name)

# Get details about the current state
current_state_df = session.sql(f"""
        SELECT OBJECT_CONSTRUCT(
            'current_user', CURRENT_USER(),
            'current_role', CURRENT_ROLE(),
            'current_secondary_roles', PARSE_JSON(CURRENT_SECONDARY_ROLES()),
            'current_database', CURRENT_DATABASE(),
            'current_schema', CURRENT_SCHEMA()
        )::STRING AS session_context;
    """).collect()

logger.info(f"Begin executing notebook {notebook_name}", extra = {'logger_name': logger_name})
logger.info(f"Using parameters database: {database_name}, schema: {schema_name}, role: {role_name}", extra = {'logger_name': logger_name})
logger.info(f"Using session context {current_state_df[0]['SESSION_CONTEXT']}", extra = {'logger_name': logger_name})

In [ ]:
!pip install openpyxl

In [ ]:
%%sql -r dataframe_1
-- Temporary solution to load in the metadata, this should be replaced with a directy query to a directory table (or a metadata table)
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/order_detail.xlsx' AS STAGE_FILE_PATH, 'order_detail' AS WORKSHEET_NAME, 'ORDER_DETAIL' AS TARGET_TABLE
UNION
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/location.xlsx', 'location', 'LOCATION';

## Create a function to load Excel worksheet to table

Create a reusable function to load an Excel worksheet to a table in Snowflake.

Note: Until we can use scoped URLs in Notebooks, via the `BUILD_SCOPED_FILE_URL()` function, we need to temporarily copy the file to a temp stage and then process from there.

In [ ]:
from snowflake.snowpark.files import SnowflakeFile
import pandas as pd
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import re

def _col_index(col_str):
    """Convert Excel column letters to 0-based index (e.g. 'A'->0, 'B'->1, 'AA'->26)."""
    result = 0
    for c in col_str:
        result = result * 26 + (ord(c) - ord('A') + 1)
    return result - 1

def parse_xlsx_worksheet(file_bytes, worksheet_name):
    """Parse an xlsx worksheet using only stdlib (zipfile + xml)."""
    ns = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
          'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'}

    with zipfile.ZipFile(BytesIO(file_bytes)) as zf:
        # Load shared strings
        shared_strings = []
        if 'xl/sharedStrings.xml' in zf.namelist():
            ss_tree = ET.parse(zf.open('xl/sharedStrings.xml'))
            for si in ss_tree.findall('.//s:si', ns):
                text_parts = si.findall('.//s:t', ns)
                shared_strings.append(''.join(t.text or '' for t in text_parts))

        # Find the sheet by name from workbook.xml
        wb_tree = ET.parse(zf.open('xl/workbook.xml'))
        sheet_elem = None
        for sheet in wb_tree.findall('.//s:sheets/s:sheet', ns):
            if sheet.get('name') == worksheet_name:
                sheet_elem = sheet
                break
        if sheet_elem is None:
            raise ValueError(f"Worksheet '{worksheet_name}' not found")

        sheet_id = sheet_elem.get('sheetId')
        r_id = sheet_elem.get(f'{{{ns["r"]}}}id')

        # Resolve rId to file path via relationships
        rels_tree = ET.parse(zf.open('xl/_rels/workbook.xml.rels'))
        sheet_file = None
        for rel in rels_tree.findall('.//{http://schemas.openxmlformats.org/package/2006/relationships}Relationship'):
            if rel.get('Id') == r_id:
                sheet_file = 'xl/' + rel.get('Target')
                break
        if sheet_file is None:
            sheet_file = f'xl/worksheets/sheet{sheet_id}.xml'

        # Parse the sheet
        sheet_tree = ET.parse(zf.open(sheet_file))
        col_re = re.compile(r'^([A-Z]+)')
        
        # First pass: determine the number of columns from the header row
        all_rows = sheet_tree.findall('.//s:sheetData/s:row', ns)
        if not all_rows:
            return pd.DataFrame()
        
        # Determine max columns from header row
        num_cols = 0
        for row_elem in all_rows:
            for cell in row_elem.findall('s:c', ns):
                ref = cell.get('r', '')
                match = col_re.match(ref)
                if match:
                    col_idx = _col_index(match.group(1))
                    num_cols = max(num_cols, col_idx + 1)
        
        # Second pass: parse all rows with correct column placement
        rows = []
        for row_elem in all_rows:
            row_data = [None] * num_cols
            for cell in row_elem.findall('s:c', ns):
                ref = cell.get('r', '')
                match = col_re.match(ref)
                if not match:
                    continue
                col_idx = _col_index(match.group(1))
                
                value = cell.find('s:v', ns)
                cell_type = cell.get('t')
                if value is not None and value.text is not None:
                    if cell_type == 's':
                        row_data[col_idx] = shared_strings[int(value.text)]
                    elif cell_type == 'b':
                        row_data[col_idx] = bool(int(value.text))
                    else:
                        try:
                            row_data[col_idx] = float(value.text) if '.' in value.text else int(value.text)
                        except ValueError:
                            row_data[col_idx] = value.text
            rows.append(row_data)

    # First row is header
    columns = rows[0] if rows else []
    data = rows[1:] if len(rows) > 1 else []
    return pd.DataFrame(data, columns=columns)

# 1. Create a temp internal stage (once at the start)
session.sql("CREATE TEMP STAGE IF NOT EXISTS temp_excel_stage").collect()

def load_excel_worksheet_to_table(session, external_path, worksheet_name, target_table):
    """Load an Excel worksheet by copying to internal stage first"""
    
    # Extract filename from path
    filename = external_path.split('/')[-1]
    
    # 2. Copy file from external to internal stage
    session.sql(f"""
        COPY FILES INTO @temp_excel_stage
        FROM {external_path}
    """).collect()
    
    # 3. Read and parse the Excel file using stdlib
    with SnowflakeFile.open(f'@temp_excel_stage/{filename}', 'rb') as f:
        file_bytes = f.read()
    
    df = parse_xlsx_worksheet(file_bytes, worksheet_name)
    
    # Write to Snowflake table
    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("overwrite").save_as_table(target_table)
    
    logger.info(f"Loaded {len(df)} rows from '{worksheet_name}' to {target_table}", extra = {'logger_name': logger_name})

## Process all Excel worksheets

Loop through each Excel worksheet to process and call our `load_excel_worksheet_to_table_local()` function.

In [ ]:
import pandas as pd

# Process each file from the sql_get_spreadsheets cell above
files_to_load = dataframe_1.copy() if isinstance(dataframe_1, pd.DataFrame) else dataframe_1.to_pandas()
for index, excel_file in files_to_load.iterrows():
    print(f"Processing Excel file {excel_file['STAGE_FILE_PATH']}")
    load_excel_worksheet_to_table(session, excel_file['STAGE_FILE_PATH'], excel_file['WORKSHEET_NAME'], excel_file['TARGET_TABLE'])

logger.info(f"Finish executing notebook {notebook_name}", extra = {'logger_name': logger_name})

### Debugging

In [ ]:
%%sql -r dataframe_2
--DESCRIBE TABLE LOCATION;
--SELECT * FROM LOCATION;
--SHOW TABLES;